# Team 1 – AI-Enhanced Network Anomaly Detection

## Session 03: Baseline Model Development

**Author:** Jennifer Silva  
**Date:** October 6, 2026

### Purpose

This notebook establishes the baseline machine-learning models for the CICIDS2017 intrusion-detection dataset. It builds on the team's completed data audit and preprocessing work by using the approved training and validation datasets.

The initial models evaluated are:

- Decision Tree
- Random Forest

The models will be trained using the 70% training dataset and evaluated against the 15% validation dataset. Performance will be compared using precision, recall, F1-score, confusion matrices, false-positive behavior, and false-negative behavior.

The separate 15% test dataset will remain untouched during model development and will be reserved for final evaluation.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
DATA_DIR = Path("../data/processed")

TRAIN_FILE = DATA_DIR / "training_dataset_CIC_70.csv"
VAL_FILE = DATA_DIR / "validation_dataset_CIC_15.csv"

print("Training dataset:", TRAIN_FILE)
print("Validation dataset:", VAL_FILE)

assert TRAIN_FILE.exists(), f"Training dataset not found: {TRAIN_FILE}"
assert VAL_FILE.exists(), f"Validation dataset not found: {VAL_FILE}"

print("\nTraining and validation datasets found successfully.")

Training dataset: ../data/processed/training_dataset_CIC_70.csv
Validation dataset: ../data/processed/validation_dataset_CIC_15.csv

Training and validation datasets found successfully.


## 1. Load and Validate Team Datasets

The training and validation datasets were generated during the team's preprocessing phase using a stratified 70/15/15 split. Before model development, the datasets are validated to confirm their dimensions, feature consistency, target variable, and data quality.

In [3]:
print("Loading training dataset...")
train_df = pd.read_csv(TRAIN_FILE, low_memory=False)

print("Loading validation dataset...")
val_df = pd.read_csv(VAL_FILE, low_memory=False)

print("\nDatasets loaded successfully.")
print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Loading training dataset...
Loading validation dataset...

Datasets loaded successfully.
Training shape: (1764525, 53)
Validation shape: (378113, 53)


In [4]:
TARGET = "Attack Type"

print("Target column present in training:", TARGET in train_df.columns)
print("Target column present in validation:", TARGET in val_df.columns)

print("\nSame columns in both datasets:",
      train_df.columns.tolist() == val_df.columns.tolist())

print("\nNumber of training features:", len(train_df.columns) - 1)
print("Number of validation features:", len(val_df.columns) - 1)

print("\nTraining target classes:")
print(train_df[TARGET].value_counts())

Target column present in training: True
Target column present in validation: True

Same columns in both datasets: True

Number of training features: 52
Number of validation features: 52

Training target classes:
Attack Type
Normal Traffic    1466539
DoS                135621
DDoS                89610
Port Scanning       63486
Brute Force          6405
Web Attacks          1500
Bots                 1364
Name: count, dtype: int64


### Pre-Model Data Quality Check

Before training the baseline models, the training and validation datasets are checked for missing values, infinite numeric values, duplicate records, and unexpected non-numeric model features. These checks help ensure that model performance is not affected by unresolved preprocessing issues.

In [5]:
# Missing values
train_missing = train_df.isna().sum().sum()
val_missing = val_df.isna().sum().sum()

# Separate model features from target
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

# Check feature data types
non_numeric_train = X_train.select_dtypes(exclude=np.number).columns.tolist()
non_numeric_val = X_val.select_dtypes(exclude=np.number).columns.tolist()

# Check infinity only in numeric model features
train_inf = np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()
val_inf = np.isinf(X_val.select_dtypes(include=np.number)).sum().sum()

# Duplicate rows
train_duplicates = train_df.duplicated().sum()
val_duplicates = val_df.duplicated().sum()

print("PRE-MODEL DATA QUALITY CHECK")
print("-" * 35)

print("Training missing values:", train_missing)
print("Validation missing values:", val_missing)

print("\nTraining infinite values:", train_inf)
print("Validation infinite values:", val_inf)

print("\nTraining duplicate rows:", train_duplicates)
print("Validation duplicate rows:", val_duplicates)

print("\nNon-numeric training features:", non_numeric_train)
print("Non-numeric validation features:", non_numeric_val)

PRE-MODEL DATA QUALITY CHECK
-----------------------------------
Training missing values: 0
Validation missing values: 0

Training infinite values: 0
Validation infinite values: 0

Training duplicate rows: 82
Validation duplicate rows: 4

Non-numeric training features: []
Non-numeric validation features: []


### Duplicate and Split Leakage Review

A small number of duplicate records remain within the processed training and validation datasets. Because the datasets were created using a random stratified split, an additional check is performed to determine whether identical feature records appear in both the training and validation sets.

Cross-split duplicates may introduce data leakage because the model could be evaluated on observations identical to records it encountered during training.

In [6]:
# Create unique feature-only records for the overlap check
train_unique = X_train.drop_duplicates()
val_unique = X_val.drop_duplicates()

# Find identical feature rows appearing in both datasets
overlap = train_unique.merge(
    val_unique,
    how="inner",
    on=X_train.columns.tolist()
)

print("Unique training feature rows:", len(train_unique))
print("Unique validation feature rows:", len(val_unique))
print("Identical feature rows across train and validation:", len(overlap))

Unique training feature rows: 1764095
Unique validation feature rows: 378094
Identical feature rows across train and validation: 190


## 2. Decision Tree Baseline Model

A Decision Tree classifier is trained using the team's cleaned CICIDS2017 training dataset. The model predicts one of seven traffic classes. Validation predictions are evaluated using class-level precision, recall, F1-score, and confusion matrices.

The training and validation datasets remain unchanged. The reserved test dataset is excluded from model development.

In [ ]:
import sklearn
import sys
import platform

print("Python:", sys.version.split()[0])
print("Scikit-learn:", sklearn.__version__)
print("Jupyter Notebook: 7.6.3")
print("Operating system:", platform.platform())

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    max_depth=15,
    min_samples_leaf=10,
    random_state=42
)

print("Training Decision Tree...")
decision_tree.fit(X_train, y_train)

print("Training completed.")

y_pred_dt = decision_tree.predict(X_val)

print("Validation predictions completed.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix
)

print("DECISION TREE VALIDATION RESULTS")
print("-" * 40)

print("Accuracy:", accuracy_score(y_val, y_pred_dt))
print("Macro F1:", f1_score(y_val, y_pred_dt, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred_dt, average="weighted"))

print("\nPer-class performance:")
print(classification_report(
    y_val,
    y_pred_dt,
    digits=4,
    zero_division=0
))


In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

classes = decision_tree.classes_

class_matrices = multilabel_confusion_matrix(
    y_val,
    y_pred_dt,
    labels=classes
)

results = []

for label, matrix in zip(classes, class_matrices):
    tn, fp, fn, tp = matrix.ravel()

    results.append({
        "Class": label,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "FPR": fp / (fp + tn) if (fp + tn) else 0,
        "FNR": fn / (fn + tp) if (fn + tp) else 0
    })

dt_error_df = pd.DataFrame(results)

display(dt_error_df)

In [ ]:
normal_label = "Normal Traffic"

actual_attack = y_val != normal_label
predicted_attack = y_pred_dt != normal_label

tn, fp, fn, tp = confusion_matrix(
    actual_attack,
    predicted_attack,
    labels=[False, True]
).ravel()

print("BINARY ATTACK DETECTION")
print("True positives:", tp)
print("False positives:", fp)
print("False negatives:", fn)
print("True negatives:", tn)

print("False-positive rate:", fp / (fp + tn))
print("False-negative rate:", fn / (fn + tp))

In [1]:
print("Kernel restarted successfully")

Kernel restarted successfully


In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/processed")

train_df = pd.read_csv(
    DATA_DIR / "training_dataset_CIC_70.csv",
    low_memory=False
)

val_df = pd.read_csv(
    DATA_DIR / "validation_dataset_CIC_15.csv",
    low_memory=False
)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)

Training dataset: (1764525, 53)
Validation dataset: (378113, 53)


In [3]:
from sklearn.model_selection import train_test_split

TARGET = "Attack Type"

# Separate features and labels
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

# Use 10% of the official training data for the initial experiment
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.10,
    stratify=y_train,
    random_state=42
)

print("Full training records:", len(X_train))
print("Sample training records:", len(X_train_sample))
print("Validation records:", len(X_val))

print("\nSample class distribution:")
print(y_train_sample.value_counts())

Full training records: 1764525
Sample training records: 176452
Validation records: 378113

Sample class distribution:
Attack Type
Normal Traffic    146653
DoS                13562
DDoS                8961
Port Scanning       6349
Brute Force          641
Web Attacks          150
Bots                 136
Name: count, dtype: int64


In [4]:
from sklearn.tree import DecisionTreeClassifier
import time

# Create Decision Tree model
decision_tree = DecisionTreeClassifier(
    max_depth=15,
    min_samples_leaf=10,
    random_state=42
)

# Train using the stratified sample
print("Starting Decision Tree training...")

start_time = time.time()

decision_tree.fit(X_train_sample, y_train_sample)

training_time = time.time() - start_time

print("Decision Tree training completed!")
print(f"Training time: {training_time:.2f} seconds")
print(f"Training records: {len(X_train_sample):,}")
print(f"Features used: {X_train_sample.shape[1]}")
print(f"Tree depth: {decision_tree.get_depth()}")
print(f"Leaf nodes: {decision_tree.get_n_leaves()}")

Starting Decision Tree training...
Decision Tree training completed!
Training time: 3.58 seconds
Training records: 176,452
Features used: 52
Tree depth: 15
Leaf nodes: 178


In [5]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix
)

# Generate predictions on the official validation dataset
y_pred_dt = decision_tree.predict(X_val)

print("=== DECISION TREE VALIDATION RESULTS ===")
print(f"Validation records: {len(y_val):,}")

print(f"\nAccuracy: {accuracy_score(y_val, y_pred_dt):.4f}")
print(f"Macro F1: {f1_score(y_val, y_pred_dt, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_val, y_pred_dt, average='weighted'):.4f}")

print("\n=== PERFORMANCE BY TRAFFIC CLASS ===")
print(classification_report(
    y_val,
    y_pred_dt,
    digits=4,
    zero_division=0
))

=== DECISION TREE VALIDATION RESULTS ===
Validation records: 378,113

Accuracy: 0.9971
Macro F1: 0.8568
Weighted F1: 0.9968

=== PERFORMANCE BY TRAFFIC CLASS ===
                precision    recall  f1-score   support

          Bots     0.4651    0.0685    0.1194       292
   Brute Force     0.9905    0.9913    0.9909      1372
          DDoS     0.9986    0.9975    0.9980     19202
           DoS     0.9949    0.9883    0.9916     29062
Normal Traffic     0.9977    0.9990    0.9983    314259
 Port Scanning     0.9891    0.9960    0.9925     13604
   Web Attacks     0.9400    0.8758    0.9068       322

      accuracy                         0.9971    378113
     macro avg     0.9109    0.8452    0.8568    378113
  weighted avg     0.9968    0.9971    0.9968    378113

